In [53]:
import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [54]:
df = pd.read_csv("../data/raw/housing.csv")

In [55]:
df["rooms_per_household"] = df["total_rooms"] / df["households"]
df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]

df["rooms_per_household"] = df["rooms_per_household"].clip(0, 20)
df["bedrooms_per_room"] = df["bedrooms_per_room"].clip(0, 1)

for col in ["total_rooms", "total_bedrooms", "population", "households", "median_income"]:
    df[col + "_log1p"] = np.log1p(df[col])

df = df.drop(columns=[
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income"
])

In [56]:
TARGET = "median_house_value"

X = df.drop(columns=[TARGET])
y = np.log1p(df[TARGET])

In [57]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(X_train.shape, X_val.shape, X_test.shape)

(14448, 11) (3096, 11) (3096, 11)


In [58]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_features = ["ocean_proximity"]

In [59]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [60]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [61]:
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

print(X_train_proc.shape)

(14448, 15)


In [63]:
with open("../models/preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)

FINAL_NUMERIC_FEATURES = list(numeric_features)
FINAL_CATEGORICAL_FEATURES = list(categorical_features)

with open("../models/feature_order.pkl", "wb") as f:
    pickle.dump(
        {
            "numeric": FINAL_NUMERIC_FEATURES,
            "categorical": FINAL_CATEGORICAL_FEATURES
        },
        f
    )